<a href="https://colab.research.google.com/github/TK-Problem/random-experiments/blob/main/001_bayesian_linear_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

## Configuration

Tickers, date range, lag, and split settings used throughout the experiment.

In [ ]:
TARGET    = "SPY"
PREDICTORS = [
    "QQQ",   # Nasdaq 100
    "IWM",   # Russell 2000 (small cap)
    "GLD",   # Gold
    "TLT",   # Long-term Treasuries
    "VIX",   # Volatility index (^VIX in yfinance)
    "XLE",   # Energy sector
    "XLF",   # Financials sector
    "EEM",   # Emerging markets
]

START_DATE = "2018-01-01"
END_DATE   = "2024-12-31"
LAG        = 1          # Days to lag predictors (1 = previous day)
TEST_SIZE  = 0.2
RANDOM_SEED = 42


## Load Data

Download daily closing prices from Yahoo Finance and compute log returns.

In [ ]:
print("Downloading price data from Yahoo Finance...")
tickers = [TARGET] + PREDICTORS
# ^VIX needs special handling
yf_tickers = [t if t != "VIX" else "^VIX" for t in tickers]

raw = yf.download(yf_tickers, start=START_DATE, end=END_DATE, auto_adjust=True)["Close"]

# Rename ^VIX back to VIX
raw.columns = [c.replace("^", "") for c in raw.columns]

# Compute daily log returns
returns = np.log(raw / raw.shift(1)).dropna()
print(f"Downloaded {len(returns)} trading days of data.\n")

## Feature Engineering

Lag predictor returns by `LAG` days so features only use information available before the target date.

In [ ]:
# Target: SPY return on day t
y_series = returns[TARGET].copy()

# Features: each predictor's return on day t-LAG
X_df = returns[PREDICTORS].shift(LAG)

# Align and drop NaNs
df = pd.concat([y_series.rename("target"), X_df], axis=1).dropna()

y = df["target"].values
X_raw = df[PREDICTORS].values

print(f"Feature matrix shape: {X_raw.shape}")
print(f"Target vector shape:  {y.shape}\n")

## Train / Test Split

Standardize features and split chronologically (no shuffle) to avoid look-ahead bias.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Train/test split (no shuffle — time series!)
split_idx = int(len(y) * (1 - TEST_SIZE))
X_train, X_test = X_scaled[:split_idx], X_scaled[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
dates_test = df.index[split_idx:]

## Bayesian Linear Regression Model

Define weakly informative priors and fit a PyMC Bayesian linear regression via NUTS sampling.

In [ ]:
n_features = X_train.shape[1]

with pm.Model() as model:

    # --- Priors ---
    # Weakly informative priors — daily returns are small
    alpha  = pm.Normal("alpha", mu=0, sigma=0.01)            # intercept
    betas  = pm.Normal("betas", mu=0, sigma=0.05,            # coefficients
                        shape=n_features)
    sigma  = pm.HalfNormal("sigma", sigma=0.01)              # noise

    # --- Linear predictor ---
    mu = alpha + pm.math.dot(X_train, betas)

    # --- Likelihood ---
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y_train)

    # --- Sample posterior ---
    print("Sampling posterior (this may take a minute)...")
    trace = pm.sample(
        draws=2000,
        tune=1000,
        chains=4,
        target_accept=0.9,
        random_seed=RANDOM_SEED,
        progressbar=True,
    )

print("\nSampling complete!")

# ============================================================
# 6. POSTERIOR SUMMARY
# ============================================================

print("\n--- Posterior Summary ---")
summary = az.summary(trace, var_names=["alpha", "betas", "sigma"], round_to=6)
summary.index = (
    ["alpha"]
    + [f"beta_{p}" for p in PREDICTORS]
    + ["sigma"]
)
print(summary)

## In-Sample Diagnostics

Trace plots and forest plot to check sampler convergence and posterior coefficient estimates.

In [ ]:
az.plot_trace(trace, var_names=["alpha", "betas", "sigma"], compact=True)
plt.suptitle("Trace Plots", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("trace_plots.png", dpi=100, bbox_inches="tight")
plt.show()

az.plot_forest(trace, var_names=["betas"],
               combined=True, hdi_prob=0.94,
               r_hat=True)
plt.axvline(0, color="red", linestyle="--", alpha=0.6)
plt.yticks(ticks=range(n_features), labels=PREDICTORS[::-1])
plt.title("Beta Coefficients — 94% HDI")
plt.tight_layout()
plt.show()

## Out-of-Sample Prediction

Compute posterior predictive means and 94% HDI on the held-out test set, then report R², RMSE, and correlation.

In [ ]:
with model:
    pm.set_data({})          # no new data nodes — use posterior directly
    # Manually compute posterior predictive on test set
    post_alpha = trace.posterior["alpha"].values.flatten()        # (chains*draws,)
    post_betas = trace.posterior["betas"].values.reshape(-1, n_features)
    post_sigma = trace.posterior["sigma"].values.flatten()

    # Predicted mean for each posterior sample: (n_samples, n_test)
    mu_pred = post_alpha[:, None] + post_betas @ X_test.T

    y_pred_mean = mu_pred.mean(axis=0)
    y_pred_hdi  = az.hdi(mu_pred, hdi_prob=0.94)

# ---- Metrics ----
ss_res = np.sum((y_test - y_pred_mean) ** 2)
ss_tot = np.sum((y_test - y_test.mean()) ** 2)
r2     = 1 - ss_res / ss_tot
rmse   = np.sqrt(np.mean((y_test - y_pred_mean) ** 2))
corr   = np.corrcoef(y_test, y_pred_mean)[0, 1]

print(f"\n--- Out-of-Sample Metrics ---")
print(f"  R²    : {r2:.4f}")
print(f"  RMSE  : {rmse:.6f}")
print(f"  Corr  : {corr:.4f}")

## Visualize Predictions

Plot actual vs. predicted SPY log returns and residuals over the test period.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(dates_test, y_test,       label="Actual SPY return",    alpha=0.7, linewidth=0.8)
axes[0].plot(dates_test, y_pred_mean,  label="Predicted (posterior mean)", alpha=0.9, linewidth=0.8)
axes[0].fill_between(dates_test,
                     y_pred_hdi[:, 0], y_pred_hdi[:, 1],
                     alpha=0.2, label="94% HDI")
axes[0].axhline(0, color="black", linewidth=0.5, linestyle="--")
axes[0].set_title(f"SPY Daily Return Prediction (Test Set)  |  R²={r2:.4f}  RMSE={rmse:.6f}")
axes[0].legend(fontsize=9)
axes[0].set_ylabel("Log Return")

residuals = y_test - y_pred_mean
axes[1].bar(dates_test, residuals, width=1, alpha=0.6, color="steelblue", label="Residuals")
axes[1].axhline(0, color="red", linewidth=0.8)
axes[1].set_title("Residuals")
axes[1].set_ylabel("Error")
axes[1].set_xlabel("Date")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## Coefficient Importance

Rank predictors by the Bayesian signal-to-noise ratio (|mean| / std) and plot posterior beta estimates.

In [ ]:
beta_means = post_betas.mean(axis=0)
beta_stds  = post_betas.std(axis=0)

coef_df = pd.DataFrame({
    "ETF":        PREDICTORS,
    "beta_mean":  beta_means,
    "beta_std":   beta_stds,
    "signal":     np.abs(beta_means) / beta_stds,   # Bayes analog of t-stat
}).sort_values("signal", ascending=False)

print("\n--- Coefficient Summary (sorted by |mean/std|) ---")
print(coef_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#e74c3c" if v > 0 else "#2980b9" for v in coef_df["beta_mean"]]
ax.barh(coef_df["ETF"], coef_df["beta_mean"], xerr=coef_df["beta_std"],
        color=colors, capsize=4, alpha=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Posterior Beta Coefficients (mean ± 1 std)")
ax.set_xlabel("Coefficient value")
plt.tight_layout()
plt.show()